In [4]:
import asyncio
async def api1():
    await asyncio.sleep(2)
    print("API 1 finished")
async def api2():
    await asyncio.sleep(2)
    print("API 2 finished")
async def main():
    t1=asyncio.create_task(api1())
    t2=asyncio.create_task(api2())
    await t1
    await t2
await main()


API 1 finished
API 2 finished


In [4]:
import asyncio
import time
async def task(name):
    await asyncio.sleep(2)
    print(name)
async def main():
    start=time.time()
    await task("Task 1")
    await task("Task 2")
    await task("Task 3")
    print("sequential:",time.time()-start)
    start=time.time()
    await asyncio.gather(task("Task 1"),
                         task("Task 2"),
                         task("Task 3"))
    print("concurrent:",time.time()-start)
await main()
    

Task 1
Task 2
Task 3
sequential: 6.01069712638855
Task 1
Task 2
Task 3
concurrent: 2.0013463497161865


In [7]:
import asyncio
async def success():
    return "success"
async def failure():
    raise Exception("error occured")
async def main():
    results=await asyncio.gather(success(),failure(),success(),return_exceptions=True)
    print(results)
await main()

['success', Exception('error occured'), 'success']


In [8]:
import asyncio
async def download(name,sec):
    await asyncio.sleep(sec)
    return name
async def main():
    tasks=[
        asyncio.create_task(download("File 1",1)),
        asyncio.create_task(download("File 2",3)),
        asyncio.create_task(download("File 3",3))
    ]
    done,pending=await asyncio.wait(tasks,timeout=2)
    print("Completed:")
    for d in done:
        print(d.result())
    for p in pending:
        p.cancel()
await main()

Completed:
File 1


In [10]:
import asyncio
async def request(name,delay):
    await asyncio.sleep(delay)
    return name
async def main():
    tasks=[
        request("A",3),
        request("B",1),
        request("C",2)
    ]
    for t in asyncio.as_completed(tasks):
        print(await t)
await main()

B
C
A


In [11]:
import asyncio
async def database():
    await asyncio.sleep(5)
async def main():
    try:
        await asyncio.wait_for(database(),timeout=2)
    except asyncio.TimeoutError:
        print("Database timeout")
await main()

Database timeout


In [12]:
import asyncio
async def email():
    try:
        while True:
            print("Sending...")
            await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("Task Cancelled")
async def main():
    task = asyncio.create_task(email())
    await asyncio.sleep(2)
    task.cancel()
    await task
await main()

Sending...
Sending...
Task Cancelled


In [13]:
import asyncio
async def calculate():
    return 100
def callback(task):
    print("Result:", task.result())
async def main():
    task = asyncio.create_task(calculate())
    task.add_done_callback(callback)
    await task

await main()

Result: 100


In [14]:
import asyncio
balance = 100
lock = asyncio.Lock()
async def update(amount):
    global balance
    async with lock:
        temp = balance
        await asyncio.sleep(1)
        balance = temp + amount
        print(balance)
async def main():
    await asyncio.gather(
        update(50),
        update(30),
        update(20)
    )

await main()

150
180
200


In [15]:
import asyncio
async def producer(queue):
    for i in range(1,11):
        await queue.put(i)
async def consumer(queue):
    while True:
        item = await queue.get()
        print("Processed:", item)
        queue.task_done()
async def main():
    queue = asyncio.Queue()
    c1 = asyncio.create_task(consumer(queue))
    c2 = asyncio.create_task(consumer(queue))
    await producer(queue)
    await queue.join()
    c1.cancel()
    c2.cancel()

await main()

Processed: 1
Processed: 2
Processed: 3
Processed: 4
Processed: 5
Processed: 6
Processed: 7
Processed: 8
Processed: 9
Processed: 10
